[//]: # (cr:doc name='chapter_c04_snapshot_and_dashboard' id=e5186854)
# Chapter c04: Snapshot + Dashboard (Causal Track)

Builds the per-scoring-run `eligibility_snapshot` table, publishes the dashboard SQL views, and prints the four-way anchor tuple in force.

**Phase 3 placeholder.** Cells 1 and 2 are gated on `RUN_PHASE3`; while the flag is `False` they print a status line and the run-summary cell at the bottom still executes against any populated `archetype_catalog`. Flip `RUN_PHASE3=True` once `snapshot_writer.py` and `dashboard_views.sql` ship in Phase 3.

Reads `predictions` from `s10_batch_inference` (do **not** trigger scoring here).


In [ ]:
# @cr:code name='init_progress' id=fbcfe04e
from customer_retention.analysis.notebook_progress import accept_workflow_params, track_and_export_previous

accept_workflow_params()
track_and_export_previous("c04_snapshot_and_dashboard.ipynb")
# --- cr:profiler ---
if __import__('os').environ.get("CR_BATCH_EXECUTION") == "1":
    import json as _j
    import os as _os
    import re as _r
    _cr_nb = _os.path.splitext(_os.path.basename(_os.environ.get("PAPERMILL_OUTPUT_PATH", "")))[0]
    if _cr_nb:
        _cr_mp = _os.path.join(_os.getcwd(), f".cr_cell_metrics_{_cr_nb}.jsonl")
        open(_cr_mp, 'w').close()
        _cr_re = _r.compile(r"^#\s*@cr:\w+\s+name='([^']+)'\s+id=(\w+)")
        def _cr_jc():
            return -1
        try:
            _s = __import__('pyspark.sql', fromlist=['SparkSession']).SparkSession.getActiveSession()
            if _s:
                def _cr_jc():  # noqa: F811
                    return _s._jsc.sc().dagScheduler().nextJobId().get()
        except Exception:
            pass
        def _cr_pre(info):
            info._cr_sj = _cr_jc()
        def _cr_post(r):
            sj = getattr(r.info, '_cr_sj', -1)
            sa = _cr_jc()
            m = _cr_re.match((r.info.raw_cell or '').split('\n')[0])
            if m:
                with open(_cr_mp, 'a') as f:
                    f.write(_j.dumps({"cell_name": m.group(1), "cell_id": m.group(2),
                                      "spark_jobs": (sa - sj) if sj >= 0 and sa >= 0 else None}) + '\n')
        get_ipython().events.register('pre_run_cell', _cr_pre)
        get_ipython().events.register('post_run_cell', _cr_post)
# --- /cr:profiler ---


[//]: # (cr:doc name='c04_configuration' id=0f0af33d)
## Configuration

The cell below is the only place you should need to edit.

- **`RUN_PHASE3`** — the snapshot writer and dashboard SQL views ship in Phase 3. While they are not implemented, this flag stays `False` so the placeholder cells print a status line and skip; the run-summary cell at the bottom still executes. Flip to `True` once Phase 3 ships and the snapshot writer is wired up.


In [ ]:
# @cr:config name='configuration' id=a236f890
RUN_PHASE3 = False


[//]: # (cr:doc name='c04_snapshot_and_dashboard_setup' id=56ae5456)
## 0. Setup

Resolves catalog / schema / model identifiers from `ScoringConfig` (reads the persisted Databricks init JSON on Databricks, or the local pipeline's `best_model_meta.json` for local runs). The composite-name-qualified gold features table name is derived here so the algorithmic cells stay free of path-construction logic.


In [ ]:
# @cr:code name='setup_and_resolve_model' id=c305ae14
from customer_retention.core.compat.detection import get_spark_session, is_databricks
from customer_retention.core.config import get_playbooks_dir
from customer_retention.core.config.experiments import get_experiments_dir
from customer_retention.stages.scoring import ScoringConfig

spark = get_spark_session()
PLAYBOOKS_DIR = get_playbooks_dir()

if is_databricks():
    scoring_config = ScoringConfig.from_databricks()
    CATALOG = scoring_config.catalog
    SCHEMA = scoring_config.schema
    MODEL_NAME = scoring_config.registered_model_name
    import mlflow

    mlflow_client = mlflow.tracking.MlflowClient()
    production_version = mlflow_client.get_model_version_by_alias(
        f"{CATALOG}.{SCHEMA}.{MODEL_NAME}", "production"
    )
    MODEL_VERSION = production_version.version
    MODEL_URI = f"models:/{CATALOG}.{SCHEMA}.{MODEL_NAME}@production"
else:
    scoring_config = ScoringConfig.from_local_config(get_experiments_dir())
    CATALOG = "local"
    SCHEMA = "local"
    MODEL_NAME = scoring_config.best_model_name or "local_model"
    MODEL_VERSION = "local"
    MODEL_URI = None

COMPOSITE_NAME = scoring_config.composite_name
GOLD_FEATURES_FQN = (
    f"{CATALOG}.{SCHEMA}.gold_features_{COMPOSITE_NAME}"
    if COMPOSITE_NAME
    else f"{CATALOG}.{SCHEMA}.gold_features"
)

ARCHETYPE_CATALOG_FQN = f"{CATALOG}.{SCHEMA}.archetype_catalog"
ELIGIBILITY_POLICY_FQN = f"{CATALOG}.{SCHEMA}.eligibility_policy"
PREDICTIONS_FQN = f"{CATALOG}.{SCHEMA}.predictions"

print(f"Resolved playbooks_dir: {PLAYBOOKS_DIR}")
print(f"Catalog/schema:         {CATALOG}.{SCHEMA}")
print(f"Composite name:         {COMPOSITE_NAME or '(unset)'}")
print(f"Gold features table:    {GOLD_FEATURES_FQN}")
print(f"Model URI:              {MODEL_URI or '(local)'}")
print(f"Model version:          {MODEL_VERSION}")


[//]: # (cr:doc name='c04_build_snapshot_section' id=ca917edd)
## 1. Build Eligibility Snapshot (Phase 3 — gated)


In [ ]:
# @cr:code name='build_eligibility_snapshot' id=1810f858
if not RUN_PHASE3:
    print("SKIPPED: RUN_PHASE3=False (snapshot_writer.py ships in Phase 3)")
else:
    raise NotImplementedError(
        "snapshot_writer.py ships in Phase 3 — see "
        "docs/causal_track_implementation_plan.md §Phase 3"
    )


[//]: # (cr:doc name='c04_publish_views_section' id=0230cc74)
## 2. Publish Dashboard SQL Views (Phase 3 — gated)


In [ ]:
# @cr:code name='publish_dashboard_views' id=085c6bc0
if not RUN_PHASE3:
    print("SKIPPED: RUN_PHASE3=False (dashboard_views.sql ships in Phase 3)")
else:
    raise NotImplementedError("dashboard_views.sql ships in Phase 3")


[//]: # (cr:doc name='c04_summary_section' id=9f690d4b)
## 3. Print Run Summary


In [ ]:
# @cr:code name='print_run_summary' id=af038f42
if spark is None or not spark.catalog.tableExists(ARCHETYPE_CATALOG_FQN):
    print("(no archetype_catalog yet — run c01..c03 first)")
else:
    counts = spark.sql(
        f"SELECT status, COUNT(*) AS n FROM {ARCHETYPE_CATALOG_FQN} GROUP BY status"
    ).collect()
    print("archetype_catalog row counts:")
    for row in counts:
        print(f"  {row['status']}: {row['n']}")
    print(f"Model: {MODEL_NAME} v{MODEL_VERSION}")


In [ ]:
# @cr:code name='release_stage_memory' id=4edf234d
from customer_retention.core.compat import release_stage_memory

release_stage_memory()
